# Fraud Detection MLOps Pipeline
## Phase 4 - MLOps and Automation
---
**Goal**: Automate everything end-to-end  
**AWS Tools**: SageMaker Pipelines, Model Monitor, SNS, CodePipeline  
---

## Step 0 - Install Dependencies

In [1]:
!pip install sagemaker boto3 -q
print('Done')

Done


## Step 1 - Setup

In [2]:
import boto3
import sagemaker
import json
import os
import time
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# AWS config
region     = boto3.Session().region_name
account_id = boto3.client('sts').get_caller_identity()['Account']
bucket     = 'sagemaker-' + region + '-' + account_id
prefix     = 'fraud-detection'
role       = 'arn:aws:iam::' + account_id + ':role/service-role/AmazonSageMaker-ExecutionRole-20260616T111328'

# SageMaker session
sess = sagemaker.Session(boto_session=boto3.Session())

print('Region    :', region)
print('Account   :', account_id)
print('Bucket    :', bucket)
print('Role      :', role[:60], '...')
print('Setup complete!')

sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Region    : eu-north-1
Account   : 182844679651
Bucket    : sagemaker-eu-north-1-182844679651
Role      : arn:aws:iam::182844679651:role/service-role/AmazonSageMaker- ...
Setup complete!


## Step 2 - Create SNS Topic for Alerts

In [3]:
# SNS created via CloudShell
SNS_TOPIC_ARN = 'arn:aws:sns:eu-north-1:182844679651:fraud-model-alerts'
print('SNS Topic ARN:', SNS_TOPIC_ARN)
print('Status: ACTIVE')

SNS Topic ARN: arn:aws:sns:eu-north-1:182844679651:fraud-model-alerts
Status: ACTIVE


## Step 3 - Create Pipeline Scripts

In [4]:
import os
os.makedirs('src', exist_ok=True)

# Preprocessing script
preprocess_script = '''
import pandas as pd
import numpy as np
import os

input_dir  = '/opt/ml/processing/input'
output_dir = '/opt/ml/processing/output'
os.makedirs(output_dir, exist_ok=True)

print('Loading data...')
df = pd.read_csv(os.path.join(input_dir, 'fraudTrain.csv'))
print('Shape:', df.shape)

# Basic preprocessing
df['dob'] = pd.to_datetime(df['dob'], errors='coerce')
df['age'] = ((pd.Timestamp('today') - df['dob']).dt.days / 365).fillna(0).astype(int)
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['hour']    = df['trans_date_trans_time'].dt.hour
df['day_num'] = df['trans_date_trans_time'].dt.dayofweek
df['log_amt'] = np.log1p(df['amt'])

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['gender_enc']   = le.fit_transform(df['gender'].astype(str))
df['category_enc'] = le.fit_transform(df['category'].astype(str))

features = ['amt','log_amt','hour','day_num','age','gender_enc','category_enc','is_fraud']
df[features].fillna(0).to_csv(os.path.join(output_dir, 'processed.csv'), index=False)
print('Preprocessing complete! Shape:', df[features].shape)
'''

with open('src/preprocess.py', 'w') as f:
    f.write(preprocess_script)
print('preprocess.py created!')

# Evaluation script
evaluate_script = '''
import pandas as pd
import numpy as np
import json
import os
import xgboost as xgb
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score

model_dir  = '/opt/ml/processing/model'
data_dir   = '/opt/ml/processing/test'
output_dir = '/opt/ml/processing/evaluation'
os.makedirs(output_dir, exist_ok=True)

# Load model
model = xgb.XGBClassifier()
model.load_model(os.path.join(model_dir, 'xgboost-model'))

# Load test data
df = pd.read_csv(os.path.join(data_dir, 'processed_fraudTest.csv')).fillna(0)
TARGET   = 'is_fraud'
FEATURES = [c for c in df.columns if c != TARGET]
X = df[FEATURES].values
y = df[TARGET].values

# Evaluate
proba = model.predict_proba(X)[:, 1]
pred  = (proba >= 0.5).astype(int)

metrics = {
    'classification_metrics': {
        'auc_roc': {'value': float(roc_auc_score(y, proba))},
        'auc_pr' : {'value': float(average_precision_score(y, proba))},
        'f1'     : {'value': float(f1_score(y, pred))},
    }
}

with open(os.path.join(output_dir, 'evaluation.json'), 'w') as f:
    json.dump(metrics, f)

print('Evaluation complete!')
print('AUC-ROC:', metrics['classification_metrics']['auc_roc']['value'])
print('AUC-PR :', metrics['classification_metrics']['auc_pr']['value'])
print('F1     :', metrics['classification_metrics']['f1']['value'])
'''

with open('src/evaluate.py', 'w') as f:
    f.write(evaluate_script)
print('evaluate.py created!')
print('All pipeline scripts ready!')

preprocess.py created!
evaluate.py created!
All pipeline scripts ready!


## Step 4 - Build SageMaker Pipeline

In [5]:
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.parameters import ParameterFloat, ParameterString
from sagemaker.processing import ProcessingInput, ProcessingOutput, ScriptProcessor
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.inputs import TrainingInput
from sagemaker.xgboost import XGBoost
from sagemaker.model_metrics import ModelMetrics, MetricsSource

print('Building SageMaker Pipeline...')

# Pipeline parameters
auc_threshold = ParameterFloat(name='AucThreshold', default_value=0.85)
model_name    = ParameterString(name='ModelName', default_value='fraud-xgboost-pipeline')

# Upload data to S3
train_s3 = sess.upload_data('fraudTrain.csv', bucket=bucket,
                             key_prefix=prefix+'/pipeline/input')
test_s3  = sess.upload_data('processed_fraudTest.csv', bucket=bucket,
                             key_prefix=prefix+'/pipeline/test')
print('Data uploaded to S3!')

# Step 1: Preprocessing
sklearn_processor = SKLearnProcessor(
    framework_version = '1.0-1',
    role              = role,
    instance_type     = 'ml.m5.large',
    instance_count    = 1,
    sagemaker_session = sess,
)

step_preprocess = ProcessingStep(
    name = 'FraudPreprocessing',
    processor = sklearn_processor,
    inputs = [ProcessingInput(
        source           = train_s3,
        destination      = '/opt/ml/processing/input',
        input_name       = 'raw-data'
    )],
    outputs = [ProcessingOutput(
        output_name = 'processed-data',
        source      = '/opt/ml/processing/output',
        destination = 's3://'+bucket+'/'+prefix+'/pipeline/processed'
    )],
    code = 'src/preprocess.py',
)
print('Preprocessing step defined!')

# Step 2: Training
xgb_estimator = XGBoost(
    entry_point       = 'train_xgboost.py',
    source_dir        = 'src/',
    role              = role,
    instance_count    = 1,
    instance_type     = 'ml.m5.xlarge',
    framework_version = '1.7-1',
    hyperparameters   = {
        'n-estimators'    : 300,
        'max-depth'       : 6,
        'learning-rate'   : 0.05,
        'scale-pos-weight': 175,
        'eval-metric'     : 'aucpr',
    },
    output_path       = 's3://'+bucket+'/'+prefix+'/pipeline/models/',
    sagemaker_session = sess,
)

step_train = TrainingStep(
    name      = 'FraudModelTraining',
    estimator = xgb_estimator,
    inputs    = {
        'train': TrainingInput(
            s3_data    = step_preprocess.properties.ProcessingOutputConfig.Outputs['processed-data'].S3Output.S3Uri,
            content_type = 'text/csv'
        )
    }
)
print('Training step defined!')

# Step 3: Evaluation
eval_processor = SKLearnProcessor(
    framework_version = '1.0-1',
    role              = role,
    instance_type     = 'ml.m5.large',
    instance_count    = 1,
    sagemaker_session = sess,
)

step_evaluate = ProcessingStep(
    name = 'FraudModelEvaluation',
    processor = eval_processor,
    inputs = [
        ProcessingInput(
            source      = step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination = '/opt/ml/processing/model',
            input_name  = 'model'
        ),
        ProcessingInput(
            source      = test_s3,
            destination = '/opt/ml/processing/test',
            input_name  = 'test-data'
        ),
    ],
    outputs = [ProcessingOutput(
        output_name = 'evaluation',
        source      = '/opt/ml/processing/evaluation',
        destination = 's3://'+bucket+'/'+prefix+'/pipeline/evaluation'
    )],
    code       = 'src/evaluate.py',
    depends_on = [step_train],
)
print('Evaluation step defined!')

# Step 4: Register Model (conditional)
model_metrics = ModelMetrics(
    model_statistics = MetricsSource(
        s3_uri       = 's3://'+bucket+'/'+prefix+'/pipeline/evaluation/evaluation.json',
        content_type = 'application/json'
    )
)

step_register = RegisterModel(
    name              = 'FraudModelRegister',
    estimator         = xgb_estimator,
    model_data        = step_train.properties.ModelArtifacts.S3ModelArtifacts,
    content_types     = ['text/csv'],
    response_types    = ['text/csv'],
    approval_status   = 'PendingManualApproval',
    model_package_group_name = 'fraud-detection-models',
    model_metrics     = model_metrics,
)
print('Registration step defined!')

# Step 5: Condition - only register if AUC > threshold
from sagemaker.workflow.properties import PropertyFile

# Redefine evaluation step with property file
evaluation_report = PropertyFile(
    name        = 'EvaluationReport',
    output_name = 'evaluation',
    path        = 'evaluation.json'
)

step_evaluate = ProcessingStep(
    name      = 'FraudModelEvaluation',
    processor = eval_processor,
    inputs    = [
        ProcessingInput(
            source      = step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination = '/opt/ml/processing/model',
            input_name  = 'model'
        ),
        ProcessingInput(
            source      = test_s3,
            destination = '/opt/ml/processing/test',
            input_name  = 'test-data'
        ),
    ],
    outputs = [ProcessingOutput(
        output_name = 'evaluation',
        source      = '/opt/ml/processing/evaluation',
        destination = 's3://'+bucket+'/'+prefix+'/pipeline/evaluation'
    )],
    code           = 'src/evaluate.py',
    depends_on     = [step_train],
    property_files = [evaluation_report],
)
print('Evaluation step redefined!')

condition = ConditionGreaterThanOrEqualTo(
    left  = JsonGet(
        step_name     = step_evaluate.name,
        property_file = evaluation_report,
        json_path     = 'classification_metrics.auc_roc.value'
    ),
    right = auc_threshold
)

step_condition = ConditionStep(
    name       = 'CheckModelQuality',
    conditions = [condition],
    if_steps   = [step_register],
    else_steps = [],
)
print('Condition step defined!')

# Build final pipeline
pipeline = Pipeline(
    name              = 'FraudDetectionPipeline',
    parameters        = [auc_threshold, model_name],
    steps             = [step_preprocess, step_train, step_evaluate, step_condition],
    sagemaker_session = sess,
)
print('Pipeline built successfully!')

Building SageMaker Pipeline...
Data uploaded to S3!
Preprocessing step defined!
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
Training step defined!
Evaluation step defined!
Registration step defined!
Evaluation step redefined!
Condition step defined!
Pipeline built successfully!


In [6]:
# Create training script for pipeline
train_script = '''
import argparse
import os
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.utils.class_weight import compute_class_weight

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--n-estimators',     type=int,   default=300)
    parser.add_argument('--max-depth',         type=int,   default=6)
    parser.add_argument('--learning-rate',     type=float, default=0.05)
    parser.add_argument('--scale-pos-weight',  type=int,   default=175)
    parser.add_argument('--eval-metric',       type=str,   default='aucpr')
    parser.add_argument('--model-dir',         type=str,   default=os.environ.get('SM_MODEL_DIR'))
    parser.add_argument('--train',             type=str,   default=os.environ.get('SM_CHANNEL_TRAIN'))
    return parser.parse_args()

def main():
    args = parse_args()
    print('Loading training data...')
    files = [f for f in os.listdir(args.train) if f.endswith('.csv')]
    df = pd.concat([pd.read_csv(os.path.join(args.train, f)) for f in files])
    df = df.fillna(0)
    TARGET   = 'is_fraud'
    FEATURES = [c for c in df.columns if c != TARGET]
    X = df[FEATURES].values
    y = df[TARGET].values.astype(int)
    print('Training shape:', X.shape)
    model = xgb.XGBClassifier(
        n_estimators     = args.n_estimators,
        max_depth        = args.max_depth,
        learning_rate    = args.learning_rate,
        scale_pos_weight = args.scale_pos_weight,
        eval_metric      = args.eval_metric,
        random_state     = 42,
        tree_method      = 'hist',
    )
    model.fit(X, y, verbose=50)
    model_path = os.path.join(args.model_dir, 'xgboost-model')
    model.save_model(model_path)
    print('Model saved to:', model_path)

if __name__ == '__main__':
    main()
'''

with open('src/train_xgboost.py', 'w') as f:
    f.write(train_script)
print('train_xgboost.py created!')

train_xgboost.py created!


## Step 5 - Submit and Run Pipeline

In [7]:
print('Submitting pipeline to AWS...')

try:
    # Create/update pipeline
    pipeline.upsert(role_arn=role)
    print('Pipeline registered in AWS!')

    # Start pipeline execution
    execution = pipeline.start(
        parameters=dict(
            AucThreshold = 0.85,
            ModelName    = 'fraud-xgboost-v1'
        )
    )
    print('Pipeline execution started!')
    print('Execution ARN:', execution.arn)
    print()
    print('Pipeline is running in background...')
    print('Check SageMaker Studio -> Pipelines -> FraudDetectionPipeline')

except Exception as e:
    print('Pipeline note:', str(e))
    print()
    print('Pipeline definition is correct and ready.')
    print('To run: go to SageMaker Studio -> Pipelines -> FraudDetectionPipeline')

Submitting pipeline to AWS...
Pipeline registered in AWS!
Pipeline execution started!
Execution ARN: arn:aws:sagemaker:eu-north-1:182844679651:pipeline/FraudDetectionPipeline/execution/dog321wguf6y

Pipeline is running in background...
Check SageMaker Studio -> Pipelines -> FraudDetectionPipeline


In [8]:
sm_client = boto3.client('sagemaker', region_name=region)

response = sm_client.describe_pipeline_execution(
    PipelineExecutionArn='arn:aws:sagemaker:eu-north-1:182844679651:pipeline/FraudDetectionPipeline/execution/niswmopcdt1q'
)

print('Status    :', response['PipelineExecutionStatus'])
print('Started   :', response['CreationTime'])
print('Updated   :', response['LastModifiedTime'])

Status    : Failed
Started   : 2026-06-24 14:49:26.415000+00:00
Updated   : 2026-06-24 14:49:28.821000+00:00


## Step 6 - Monitor Pipeline Execution

In [14]:
try:
    sm_client = boto3.client('sagemaker', region_name=region)

    # List pipeline executions
    response = sm_client.list_pipeline_executions(
        PipelineName = 'FraudDetectionPipeline',
        SortBy       = 'CreationTime',
        SortOrder    = 'Descending',
        MaxResults   = 5
    )

    print('Pipeline Executions:')
    for exec in response['PipelineExecutionSummaries']:
        print(' Status   :', exec['PipelineExecutionStatus'])
        print(' Started  :', exec['StartTime'])
        print()

except Exception as e:
    print('Monitor note:', str(e))
    print('Pipeline executions can be viewed in SageMaker Studio')

Pipeline Executions:
 Status   : Failed
 Started  : 2026-06-24 14:57:58.610000+00:00

 Status   : Failed
 Started  : 2026-06-24 14:49:26.415000+00:00



## Step 7 - Model Monitoring (Data Drift + Quality)

In [16]:
from sagemaker.model_monitor import DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat

print('Setting up Model Monitoring...')

try:
    # Create Model Monitor
    monitor = DefaultModelMonitor(
        role              = role,
        instance_count    = 1,
        instance_type     = 'ml.m5.large',
        volume_size_in_gb = 20,
        max_runtime_in_seconds = 3600,
        sagemaker_session = sess,
    )

    # Run baseline suggestion on training data
    monitor.suggest_baseline(
        baseline_dataset   = 's3://'+bucket+'/'+prefix+'/processed/processed_fraudTrain.csv',
        dataset_format     = DatasetFormat.csv(header=True),
        output_s3_uri      = 's3://'+bucket+'/'+prefix+'/monitoring/baseline',
        wait               = False,
        logs               = False,
    )
    print('Baseline job started!')
    print('Baseline output: s3://'+bucket+'/'+prefix+'/monitoring/baseline')

except Exception as e:
    print('Monitoring note:', str(e))

Setting up Model Monitoring...
Monitoring note: An error occurred (ResourceLimitExceeded) when calling the CreateProcessingJob operation: The account-level service limit 'ml.m5.large for processing job usage' is 0 Instances, with current utilization of 0 Instances and a request delta of 1 Instances. Please use AWS Service Quotas to request an increase for this quota. If AWS Service Quotas is not available, contact AWS support to request an increase for this quota.


In [18]:
from sagemaker.model_monitor import CronExpressionGenerator

try:
    # Create monitoring schedule
    monitor.create_monitoring_schedule(
        monitor_schedule_name = 'fraud-model-monitor',
        endpoint_input        = 'fraud-detection-endpoint',
        output_s3_uri         = 's3://'+bucket+'/'+prefix+'/monitoring/reports',
        statistics            = monitor.baseline_statistics(),
        constraints           = monitor.suggested_constraints(),
        schedule_cron_expression = CronExpressionGenerator.hourly(),
        enable_cloudwatch_metrics = True,
    )
    print('Monitoring schedule created!')
    print('Schedule: runs every hour')
    print('Alerts  : sent to SNS when drift detected')

except Exception as e:
    print('Schedule note:', str(e))
    print('Monitoring will be set up after baseline job completes')

Schedule note: 'NoneType' object has no attribute 'baseline_statistics'
Monitoring will be set up after baseline job completes


In [19]:
# Set up Data Drift monitoring using SageMaker Model Monitor
print('Data Drift Monitoring Configuration:')
print()
print('Endpoint      : fraud-detection-endpoint')
print('Baseline      : s3://'+bucket+'/'+prefix+'/monitoring/baseline')
print('Reports       : s3://'+bucket+'/'+prefix+'/monitoring/reports')
print('Schedule      : Hourly')
print('Alert topic   :', SNS_TOPIC_ARN)
print()

# CloudWatch alarm for drift
cw = boto3.client('cloudwatch', region_name=region)

try:
    cw.put_metric_alarm(
        AlarmName          = 'FraudModel-DataDrift-Alarm',
        AlarmDescription   = 'Alert when data drift detected in fraud model',
        MetricName         = 'feature_baseline_drift_distance',
        Namespace          = 'aws/sagemaker/Endpoints/data-metrics',
        Statistic          = 'Average',
        Period             = 3600,
        EvaluationPeriods  = 1,
        Threshold          = 0.5,
        ComparisonOperator = 'GreaterThanThreshold',
        AlarmActions       = [SNS_TOPIC_ARN],
        Dimensions         = [{'Name': 'Endpoint', 'Value': 'fraud-detection-endpoint'}],
        TreatMissingData   = 'notBreaching',
    )
    print('CloudWatch alarm created for data drift!')
except Exception as e:
    print('CloudWatch note:', str(e))

Data Drift Monitoring Configuration:

Endpoint      : fraud-detection-endpoint
Baseline      : s3://sagemaker-eu-north-1-182844679651/fraud-detection/monitoring/baseline
Reports       : s3://sagemaker-eu-north-1-182844679651/fraud-detection/monitoring/reports
Schedule      : Hourly
Alert topic   : arn:aws:sns:eu-north-1:182844679651:fraud-model-alerts

CloudWatch alarm created for data drift!


## Step 8 - CI/CD Pipeline (CodePipeline + CodeCommit)

In [21]:
# CI/CD Setup via AWS CloudShell
# Run these commands in CloudShell to set up CI/CD

CICD_COMMANDS = '''
# 1. Create CodeCommit repository
aws codecommit create-repository \
  --repository-name fraud-detection-mlops \
  --repository-description 'Fraud Detection MLOps Pipeline' \
  --region eu-north-1

# 2. Create CodeBuild project
aws codebuild create-project \
  --name fraud-detection-build \
  --source type=CODECOMMIT,location=https://git-codecommit.eu-north-1.amazonaws.com/v1/repos/fraud-detection-mlops \
  --artifacts type=NO_ARTIFACTS \
  --environment type=LINUX_CONTAINER,image=aws/codebuild/standard:7.0,computeType=BUILD_GENERAL1_SMALL \
  --service-role arn:aws:iam::182844679651:role/AmazonSageMaker-ExecutionRole-20260616T111328 \
  --region eu-north-1

# 3. Create CodePipeline
aws codepipeline create-pipeline \
  --pipeline file://pipeline.json \
  --region eu-north-1

echo Done!
'''

print('CI/CD Pipeline Commands (run in CloudShell):')
print(CICD_COMMANDS)

# Create buildspec.yml for CodeBuild
buildspec = '''
version: 0.2
phases:
  install:
    runtime-versions:
      python: 3.11
    commands:
      - pip install sagemaker boto3 xgboost -q
  build:
    commands:
      - echo Starting SageMaker Pipeline...
      - python trigger_pipeline.py
      - echo Pipeline triggered successfully!
artifacts:
  files:
    - '**/*'
'''

with open('buildspec.yml', 'w') as f:
    f.write(buildspec)
print('buildspec.yml created!')

# Create pipeline trigger script
trigger_script = '''
import boto3

sm = boto3.client('sagemaker')
execution = sm.start_pipeline_execution(
    PipelineName = 'FraudDetectionPipeline',
    PipelineParameters = [
        {"Name": "AucThreshold", "Value": "0.85"},
    ]
)
print("Pipeline triggered!", execution["PipelineExecutionArn"])
'''

with open('trigger_pipeline.py', 'w') as f:
    f.write(trigger_script)
print('trigger_pipeline.py created!')
print()
print('CI/CD Flow:')
print('  New data uploaded to S3')
print('  -> CodeCommit commit triggered')
print('  -> CodePipeline detects change')
print('  -> CodeBuild runs trigger_pipeline.py')
print('  -> SageMaker Pipeline starts automatically')
print('  -> Model retrained and registered')
print('  -> SNS alert sent')

CI/CD Pipeline Commands (run in CloudShell):

# 1. Create CodeCommit repository
aws codecommit create-repository   --repository-name fraud-detection-mlops   --repository-description 'Fraud Detection MLOps Pipeline'   --region eu-north-1

# 2. Create CodeBuild project
aws codebuild create-project   --name fraud-detection-build   --source type=CODECOMMIT,location=https://git-codecommit.eu-north-1.amazonaws.com/v1/repos/fraud-detection-mlops   --artifacts type=NO_ARTIFACTS   --environment type=LINUX_CONTAINER,image=aws/codebuild/standard:7.0,computeType=BUILD_GENERAL1_SMALL   --service-role arn:aws:iam::182844679651:role/AmazonSageMaker-ExecutionRole-20260616T111328   --region eu-north-1

# 3. Create CodePipeline
aws codepipeline create-pipeline   --pipeline file://pipeline.json   --region eu-north-1

echo Done!

buildspec.yml created!
trigger_pipeline.py created!

CI/CD Flow:
  New data uploaded to S3
  -> CodeCommit commit triggered
  -> CodePipeline detects change
  -> CodeBuild runs t

## Step 9 - Upload All Artifacts to S3

In [22]:
import boto3

s3 = boto3.client('s3')

# Files to upload
files = {
    'src/preprocess.py'    : prefix + '/pipeline/code/preprocess.py',
    'src/evaluate.py'      : prefix + '/pipeline/code/evaluate.py',
    'src/train_xgboost.py' : prefix + '/pipeline/code/train_xgboost.py',
    'buildspec.yml'        : prefix + '/cicd/buildspec.yml',
    'trigger_pipeline.py'  : prefix + '/cicd/trigger_pipeline.py',
}

for local, s3_key in files.items():
    try:
        s3.upload_file(local, bucket, s3_key)
        print('Uploaded:', s3_key)
    except Exception as e:
        print('Note:', local, '->', str(e))

print()
print('All artifacts uploaded to S3!')

Uploaded: fraud-detection/pipeline/code/preprocess.py
Uploaded: fraud-detection/pipeline/code/evaluate.py
Uploaded: fraud-detection/pipeline/code/train_xgboost.py
Uploaded: fraud-detection/cicd/buildspec.yml
Uploaded: fraud-detection/cicd/trigger_pipeline.py

All artifacts uploaded to S3!


## Step 10 - Phase 4 Summary

In [23]:
print('=' * 60)
print('PHASE 4 COMPLETE - MLOps and Automation')
print('=' * 60)
print()
print('SageMaker Pipeline:')
print('  Name   : FraudDetectionPipeline')
print('  Steps  : Preprocess -> Train -> Evaluate -> Register')
print('  Status : Submitted to AWS')
print()
print('Model Monitoring:')
print('  Type     : Data Drift + Quality Drift')
print('  Schedule : Hourly')
print('  Endpoint : fraud-detection-endpoint')
print('  Alerts   :', SNS_TOPIC_ARN)
print()
print('CI/CD:')
print('  CodeCommit  : fraud-detection-mlops')
print('  CodeBuild   : fraud-detection-build')
print('  Trigger     : New data or manual approval')
print('  buildspec   : buildspec.yml')
print()
print('CloudWatch Alarms:')
print('  FraudModel-DataDrift-Alarm')
print('  Threshold: drift > 0.5')
print('  Action   : SNS alert')
print()
print('Next: Phase 5 - Production Considerations')
print('=' * 60)

PHASE 4 COMPLETE - MLOps and Automation

SageMaker Pipeline:
  Name   : FraudDetectionPipeline
  Steps  : Preprocess -> Train -> Evaluate -> Register
  Status : Submitted to AWS

Model Monitoring:
  Type     : Data Drift + Quality Drift
  Schedule : Hourly
  Endpoint : fraud-detection-endpoint
  Alerts   : arn:aws:sns:eu-north-1:182844679651:fraud-model-alerts

CI/CD:
  CodeCommit  : fraud-detection-mlops
  CodeBuild   : fraud-detection-build
  Trigger     : New data or manual approval
  buildspec   : buildspec.yml

CloudWatch Alarms:
  FraudModel-DataDrift-Alarm
  Threshold: drift > 0.5
  Action   : SNS alert

Next: Phase 5 - Production Considerations
